# FedFlower Phase 2 — Federated Learning

> **Before running:** Go to `Runtime → Change Runtime Type → T4 GPU → Save`

## Cell 1 — Imports & Reload Model

Run this even if continuing from Notebook 1 in the same session — it ensures everything is defined.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision.datasets as datasets, torchvision.transforms as transforms, torchvision.models as models
from torch.utils.data import DataLoader, Subset, ConcatDataset
import numpy as np, matplotlib.pyplot as plt, copy, json, os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

class FlowerCNN(nn.Module):
    def __init__(self, num_classes=102):
        super().__init__()
        self.backbone = models.resnet50(weights='IMAGENET1K_V2')
        for name, param in self.backbone.named_parameters():
            if 'layer4' not in name and 'fc' not in name:
                param.requires_grad = False
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 512), nn.BatchNorm1d(512),
            nn.ReLU(inplace=True), nn.Dropout(0.4), nn.Linear(512, num_classes)
        )
    def forward(self, x): return self.backbone(x)

print("✅ FlowerCNN class defined")

## Cell 2 — Reload Dataset

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((256,256)), transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((224,224)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
train_data = datasets.Flowers102('./data', split='train', download=True, transform=train_transform)
val_data   = datasets.Flowers102('./data', split='val',   download=True, transform=train_transform)
test_data  = datasets.Flowers102('./data', split='test',  download=True, transform=test_transform)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2)
print(f"✅ Dataset loaded")

## Cell 3 — Split Dataset Across 5 Federated Clients

Each client gets ~408 images. They train **locally** and never share raw images — only model weights.

In [ ]:
NUM_CLIENTS = 5
all_data    = ConcatDataset([train_data, val_data])
total       = len(all_data)

np.random.seed(42)
indices        = np.random.permutation(total)
client_indices = np.array_split(indices, NUM_CLIENTS)

client_loaders = []
for i, idx in enumerate(client_indices):
    subset = Subset(all_data, idx.tolist())
    loader = DataLoader(subset, batch_size=32, shuffle=True, num_workers=2)
    client_loaders.append(loader)
    print(f"  Client {i+1}: {len(subset)} images")

print(f"\n✅ {NUM_CLIENTS} clients ready — raw images never leave their local partition!")

## Cell 4 — Define local_train() and fedavg()

In [ ]:
def local_train(global_model, loader, local_epochs=3, lr=5e-5, device='cuda'):
    """Each client trains a local copy of the global model. Returns weights only."""    local_model = copy.deepcopy(global_model)
    local_model.train()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, local_model.parameters()), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    for _ in range(local_epochs):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(local_model(imgs), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(local_model.parameters(), 1.0)
            optimizer.step()
    return local_model.state_dict()  # Only weights sent to server — never raw data

def fedavg(global_model, client_weights_list, client_sizes):
    """Weighted average of client weights. Larger clients have more influence."""    total   = sum(client_sizes)
    avg_w   = copy.deepcopy(client_weights_list[0])
    for key in avg_w:
        avg_w[key] = torch.zeros_like(avg_w[key], dtype=torch.float32)
        for i, cw in enumerate(client_weights_list):
            avg_w[key] += cw[key].float() * (client_sizes[i] / total)
    global_model.load_state_dict(avg_w)
    return global_model

def evaluate(model, loader, device):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            _, preds = model(imgs).max(1)
            correct += preds.eq(labels).sum().item(); total += labels.size(0)
    return 100.0 * correct / total

print("✅ local_train, fedavg, evaluate defined")

## Cell 5 — Run Federated Training (10 Rounds × 5 Clients)

**Expected time:** ~50 minutes. Expected final accuracy: **80–88%** (3–8% below centralized — this is normal and acceptable).

In [ ]:
GLOBAL_ROUNDS = 10
LOCAL_EPOCHS  = 3

global_model = FlowerCNN(num_classes=102).to(device)
if os.path.exists('best_model.pth'):
    global_model.load_state_dict(torch.load('best_model.pth', map_location=device))
    print("✅ Warm-started from centralized checkpoint")
else:
    print("⚠️  No best_model.pth found — starting from ImageNet weights (needs more rounds)")

fed_history = []
print(f"\nRunning {GLOBAL_ROUNDS} rounds × {NUM_CLIENTS} clients × {LOCAL_EPOCHS} local epochs")
print("=" * 60)

for round_num in range(GLOBAL_ROUNDS):
    print(f"\n🔄 Round {round_num+1}/{GLOBAL_ROUNDS}")
    client_weights_list, client_sizes = [], []

    for i, loader in enumerate(client_loaders):
        print(f"   Client {i+1} training...", end=" ", flush=True)
        w = local_train(global_model, loader, LOCAL_EPOCHS, lr=5e-5, device=device)
        client_weights_list.append(w)
        client_sizes.append(len(loader.dataset))
        print("done")

    global_model = fedavg(global_model, client_weights_list, client_sizes)
    acc = evaluate(global_model, test_loader, device)
    fed_history.append(acc)
    print(f"   ✅ Global test accuracy: {acc:.2f}%")

torch.save(global_model.state_dict(), 'federated_model.pth')
print(f"\n🎉 Federated training complete!")
print(f"   Best: {max(fed_history):.2f}%  |  Final: {fed_history[-1]:.2f}%")
print("   Download federated_model.pth from Files panel if needed")

## Cell 6 — Plot Federated vs Centralized Accuracy

In [ ]:
centralized_acc = None
if os.path.exists('centralized_results.json'):
    centralized_acc = json.load(open('centralized_results.json'))['centralized_test_acc']

plt.figure(figsize=(11, 5))
plt.plot(range(1, GLOBAL_ROUNDS+1), fed_history, marker='s', color='#E63946',
         linewidth=2, markersize=8, label='Federated CNN (FedAvg)')
if centralized_acc:
    plt.axhline(y=centralized_acc, color='#1F4E79', linewidth=2, linestyle='--',
                label=f'Centralized CNN ({centralized_acc:.1f}%)')
plt.xlabel('Federated Round'); plt.ylabel('Test Accuracy (%)')
plt.title('Federated vs Centralized — Oxford 102 Flowers', fontweight='bold')
plt.legend(); plt.grid(True, alpha=0.3); plt.xticks(range(1, GLOBAL_ROUNDS+1))
plt.tight_layout()
plt.savefig('comparison.png', dpi=150)
plt.show()

if centralized_acc:
    gap = centralized_acc - fed_history[-1]
    print(f"Centralized: {centralized_acc:.2f}%  |  Federated: {fed_history[-1]:.2f}%  |  Gap: {gap:.2f}%")
    print("✅ Gap < 8% = federated learning is working correctly!")